In [3]:
# ============================================================
# ScamShield AI — Notebook 1: Data Collection & Preparation
# ============================================================
# 
# PURPOSE: Download, explore, clean, and prepare all datasets
# for training our scam detection models.
#
# DATASETS WE WILL USE:
#   1. SMS Spam Collection (UCI) — 5,574 SMS messages
#   2. Phishing Email Dataset (Kaggle) — ~18,000 emails  
#   3. Phishing URL Dataset (Kaggle) — ~11,000 URLs
#   4. Indian Scam SMS (Custom) — we create this ourselves
# ============================================================

# Standard library imports
import os
import re
import zipfile
import urllib.request
import warnings
warnings.filterwarnings('ignore')  # Suppress minor warnings

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from collections import Counter

# Configure display settings
pd.set_option('display.max_colwidth', 100)  # Show full text in columns
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

# Configure plot styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set figure size default
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("=" * 55)
print("  ScamShield AI — Data Collection Notebook")
print("=" * 55)
print(f"  NumPy version:    {np.__version__}")
print(f"  Pandas version:   {pd.__version__}")
print(f"  Working directory: {os.getcwd()}")
print("=" * 55)

  ScamShield AI — Data Collection Notebook
  NumPy version:    1.26.3
  Pandas version:   2.1.4
  Working directory: C:\Users\Manoj S\Desktop\scamshield-ai\ml_development


In [4]:
# ============================================================
# Download NLTK resources
# ============================================================
# NLTK (Natural Language Toolkit) needs some data files 
# for tokenization, stopwords, etc.
# We download them once and they're saved locally.

print("Downloading NLTK resources...")
print("(This is a one-time download)")
print()

nltk_resources = [
    'punkt',           # Sentence/word tokenizer
    'punkt_tab',       # Updated tokenizer data
    'stopwords',       # Common words to ignore (the, is, at, etc.)
    'wordnet',         # English word database for lemmatization
    'averaged_perceptron_tagger',  # Part-of-speech tagger
    'omw-1.4',         # Open Multilingual Wordnet
]

for resource in nltk_resources:
    try:
        nltk.download(resource, quiet=True)
        print(f"  [OK] {resource}")
    except Exception as e:
        print(f"  [SKIP] {resource}: {e}")

print()
print("NLTK resources ready!")

(This is a one-time download)

  [OK] punkt
  [OK] punkt_tab
  [OK] stopwords
  [OK] wordnet
  [OK] averaged_perceptron_tagger
  [OK] omw-1.4

NLTK resources ready!


In [5]:
# ============================================================
# Directory Setup
# ============================================================
# We define all paths here so if the folder structure changes,
# we only need to update this one cell.

# Get the ml_development folder (where this notebook is)
NOTEBOOK_DIR = os.getcwd()

# Project root is one level up from ml_development
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)

# Data directories
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

# Create directories if they don't exist
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print("Directory Setup:")
print(f"  Project Root:    {PROJECT_ROOT}")
print(f"  Raw Data:        {RAW_DATA_DIR}")
print(f"  Processed Data:  {PROCESSED_DATA_DIR}")
print()

# Verify directories exist
for path in [RAW_DATA_DIR, PROCESSED_DATA_DIR]:
    exists = "EXISTS" if os.path.exists(path) else "MISSING"
    print(f"  [{exists}] {path}")

Directory Setup:
  Project Root:    C:\Users\Manoj S\Desktop\scamshield-ai
  Raw Data:        C:\Users\Manoj S\Desktop\scamshield-ai\data\raw
  Processed Data:  C:\Users\Manoj S\Desktop\scamshield-ai\data\processed

  [EXISTS] C:\Users\Manoj S\Desktop\scamshield-ai\data\raw
  [EXISTS] C:\Users\Manoj S\Desktop\scamshield-ai\data\processed


In [6]:
# ============================================================
# DATASET 1: SMS Spam Collection
# ============================================================
# Source: UCI Machine Learning Repository
# Size: 5,574 SMS messages
# Labels: 'ham' (legitimate) or 'spam' (scam)
# 
# This dataset contains real SMS messages collected for 
# spam research. It's the most widely used SMS spam dataset.
# ============================================================

def download_sms_dataset():
    """Download the SMS Spam Collection dataset."""
    
    sms_file = os.path.join(RAW_DATA_DIR, 'SMSSpamCollection')
    
    # Check if already downloaded
    if os.path.exists(sms_file):
        print("SMS dataset already exists. Skipping download.")
        return sms_file
    
    print("Downloading SMS Spam Collection dataset...")
    
    # Try primary source
    try:
        url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
        zip_path = os.path.join(RAW_DATA_DIR, 'smsspamcollection.zip')
        urllib.request.urlretrieve(url, zip_path)
        
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(RAW_DATA_DIR)
        
        os.remove(zip_path)  # Clean up zip file
        print("  Downloaded from UCI repository")
        return sms_file
        
    except Exception as e:
        print(f"  Primary source failed: {e}")
        print("  Creating dataset manually from embedded data...")
        return None

sms_path = download_sms_dataset()

# Load the SMS dataset
# The file is tab-separated with no header
# Column 1: label (ham/spam)
# Column 2: message text

if sms_path and os.path.exists(sms_path):
    sms_df = pd.read_csv(
        sms_path,
        sep='\t',           # Tab-separated
        header=None,         # No column names in file
        names=['label', 'text'],  # We name them ourselves
        encoding='latin-1'   # This encoding handles special characters
    )
    print(f"Loaded SMS dataset: {sms_df.shape[0]} rows, {sms_df.shape[1]} columns")
else:
    # Fallback: Create dataset with known good samples for demonstration
    # In real scenario, you'd manually download from UCI
    print("Creating fallback SMS dataset...")
    
    # These are example messages to demonstrate the format
    # We'll augment with Indian scam data later
    sms_data = {
        'label': ['ham', 'spam', 'ham', 'spam', 'ham', 'spam', 'ham', 'spam'],
        'text': [
            'Hey, are you coming to the party tonight?',
            'WINNER!! As a valued customer, you have been selected to receive a 900 prize. Call 09061701461 now!',
            'Can you pick me up from the station at 6pm?',
            'Free entry in 2 a weekly competition to win FA Cup final tickets. Text FA to 87121 to receive entry.',
            'I am going to be late for the meeting. Sorry!',
            'Urgent! Your account has been compromised. Click here immediately to verify: http://secure-bank.xyz',
            'The lunch was great! We should do it again.',
            'CONGRATULATIONS! You have won Rs.50000. Send your bank details to claim your prize immediately.'
        ]
    }
    sms_df = pd.DataFrame(sms_data)
    print("Fallback dataset created")

print()
print("First 5 rows:")
print(sms_df.head())
print()
print("Dataset Info:")
print(f"  Shape: {sms_df.shape}")
print(f"  Labels: {sms_df['label'].value_counts().to_dict()}")

SMS dataset already exists. Skipping download.
Loaded SMS dataset: 5572 rows, 2 columns

First 5 rows:
  label  \
0   ham   
1   ham   
2  spam   
3   ham   
4   ham   

                                                                                                  text  
0  Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there g...  
1                                                                        Ok lar... Joking wif u oni...  
2  Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive ...  
3                                                    U dun say so early hor... U c already then say...  
4                                        Nah I don't think he goes to usf, he lives around here though  

Dataset Info:
  Shape: (5572, 2)
  Labels: {'ham': 4825, 'spam': 747}


In [7]:
# ============================================================
# DATASET 2: Indian Scam SMS (Custom Dataset)
# ============================================================
# WHY CREATE THIS?
# Standard datasets are mostly US/UK focused. Indian scams have
# unique patterns:
#   - UPI/Paytm/PhonePe fraud
#   - KYC update scams
#   - SBI/HDFC/ICICI bank fraud
#   - OTP theft
#   - Lottery/prize scams in Indian context
#   - Hindi-English mixed (Hinglish) messages
#
# We create this manually and through augmentation.
# ============================================================

indian_scam_messages = [
    # ── UPI/Payment Scams ─────────────────────────────────────
    {
        'label': 'spam',
        'text': 'Your UPI ID has been blocked! Complete KYC immediately to continue transactions. Click: http://upi-kyc-verify.com',
        'category': 'upi_fraud'
    },
    {
        'label': 'spam', 
        'text': 'Aapka Paytm account band ho jayega! Abhi KYC complete karein: http://paytm-kyc.xyz/verify',
        'category': 'upi_fraud'
    },
    {
        'label': 'spam',
        'text': 'URGENT: Your PhonePe account will be deactivated in 24 hours. Update KYC now: bit.ly/phonepescam',
        'category': 'upi_fraud'
    },
    {
        'label': 'spam',
        'text': 'Dear Customer, Send Rs 1 to verify your UPI and get Rs 5000 cashback instantly! Limited offer.',
        'category': 'upi_fraud'
    },
    {
        'label': 'spam',
        'text': 'Your Google Pay account needs verification. Share OTP 847291 received on your number to continue.',
        'category': 'upi_fraud'
    },
    
    # ── Bank KYC Scams ────────────────────────────────────────
    {
        'label': 'spam',
        'text': 'Dear SBI Customer, Your account will be blocked today. Update PAN card details: http://sbi-kyc-update.com',
        'category': 'bank_fraud'
    },
    {
        'label': 'spam',
        'text': 'HDFC Bank Alert: Your debit card is blocked due to suspicious activity. Call 9876543210 immediately.',
        'category': 'bank_fraud'
    },
    {
        'label': 'spam',
        'text': 'ICICI Bank: Your account shows unusual login. Verify now or account will be suspended: http://icici-secure.xyz',
        'category': 'bank_fraud'
    },
    {
        'label': 'spam',
        'text': 'Axis Bank: Complete your pending KYC before 31st Dec or your account will be permanently closed.',
        'category': 'bank_fraud'
    },
    {
        'label': 'spam',
        'text': 'Your Aadhaar linked bank account needs immediate verification. Share OTP to prevent account freeze.',
        'category': 'bank_fraud'
    },
    
    # ── OTP Theft Scams ───────────────────────────────────────
    {
        'label': 'spam',
        'text': 'Hi I am calling from SBI. We need to verify your account. Please share the OTP you just received.',
        'category': 'otp_fraud'
    },
    {
        'label': 'spam',
        'text': 'Amazon delivery agent here. Your package needs OTP verification. Please share OTP 783291.',
        'category': 'otp_fraud'
    },
    {
        'label': 'spam',
        'text': 'IRCTC: Your ticket is confirmed. Share OTP for seat upgrade: please tell us 6-digit OTP received.',
        'category': 'otp_fraud'
    },
    
    # ── Lottery/Prize Scams ───────────────────────────────────
    {
        'label': 'spam',
        'text': 'Congratulations! Your mobile number has won Rs 25 Lakh in KBC Lucky Draw. Call KBC manager: 8800123456',
        'category': 'lottery_fraud'
    },
    {
        'label': 'spam',
        'text': 'You have been selected for Jio Lucky Customer offer! Win iPhone 15 FREE. Register now: jio-lucky.xyz',
        'category': 'lottery_fraud'
    },
    {
        'label': 'spam',
        'text': 'BSNL Lottery Result: Your number 9876543210 won Rs 10,00,000. Claim within 48 hours. Call 011-45678901',
        'category': 'lottery_fraud'
    },
    {
        'label': 'spam',
        'text': 'Amazon Great Sale Winner! You won a Samsung TV. Pay Rs 499 shipping to claim. Limited time offer!',
        'category': 'lottery_fraud'
    },
    
    # ── Job/Work From Home Scams ──────────────────────────────
    {
        'label': 'spam',
        'text': 'Earn Rs 5000 daily working from home! No experience needed. WhatsApp us: wa.me/9123456789',
        'category': 'job_fraud'
    },
    {
        'label': 'spam',
        'text': 'Part time job: Like YouTube videos and earn Rs 500 per hour. Registration fee only Rs 199.',
        'category': 'job_fraud'
    },
    {
        'label': 'spam',
        'text': 'Data entry work from home. Earn Rs 15000/month. Register on: workfromhome-india.xyz Pay Rs 999 fee.',
        'category': 'job_fraud'
    },
    
    # ── Investment/Trading Scams ──────────────────────────────
    {
        'label': 'spam',
        'text': 'Invest Rs 5000 in our crypto scheme and earn Rs 50000 in 7 days. Guaranteed returns! Join now.',
        'category': 'investment_fraud'
    },
    {
        'label': 'spam',
        'text': 'Join our SEBI registered investment group. We give 500% returns in stock market. Free tips daily.',
        'category': 'investment_fraud'
    },
    
    # ── Legitimate Indian Messages (ham) ──────────────────────
    {
        'label': 'ham',
        'text': 'Your SBI account XXXXX1234 has been credited with Rs 5000 on 15-Jan-2024. Balance: Rs 12500.',
        'category': 'legitimate_bank'
    },
    {
        'label': 'ham',
        'text': 'OTP for your SBI transaction is 847291. Valid for 10 minutes. Do not share with anyone.',
        'category': 'legitimate_otp'
    },
    {
        'label': 'ham',
        'text': 'Your IRCTC booking for Train 12345 is confirmed. PNR: 1234567890. Journey: 20-Jan-2024.',
        'category': 'legitimate_travel'
    },
    {
        'label': 'ham',
        'text': 'Swiggy: Your order #12345 has been picked up by delivery partner. Arriving in 25 minutes.',
        'category': 'legitimate_ecommerce'
    },
    {
        'label': 'ham',
        'text': 'Zomato: Your order from Pizza Hut is on the way! Track: zomato.com/track/abc123',
        'category': 'legitimate_ecommerce'
    },
    {
        'label': 'ham',
        'text': 'Kal college band hai. Faculty meeting hai. Koi classes nahi hongi. - Class Group',
        'category': 'legitimate_personal'
    },
    {
        'label': 'ham',
        'text': 'Amazon: Your order #405-1234567 has been shipped. Expected delivery: 18-Jan-2024. Track order.',
        'category': 'legitimate_ecommerce'
    },
    {
        'label': 'ham',
        'text': 'HDFC Bank: Rs 1500 debited from AC XXXX5678 at RELIANCE MART on 15-Jan-2024. Bal: Rs 23,450.',
        'category': 'legitimate_bank'
    },
]

# Create DataFrame
indian_df = pd.DataFrame(indian_scam_messages)

# Save to raw data folder
indian_csv_path = os.path.join(RAW_DATA_DIR, 'indian_scam_sms.csv')
indian_df.to_csv(indian_csv_path, index=False, encoding='utf-8')

print("Indian Scam SMS Dataset Created!")
print(f"  Total messages: {len(indian_df)}")
print(f"  Spam messages:  {len(indian_df[indian_df['label']=='spam'])}")
print(f"  Ham messages:   {len(indian_df[indian_df['label']=='ham'])}")
print()
print("Category breakdown:")
print(indian_df['category'].value_counts())
print()
print("Sample messages:")
print(indian_df[['label', 'text']].head(5).to_string())

Indian Scam SMS Dataset Created!
  Total messages: 30
  Spam messages:  22
  Ham messages:   8

Category breakdown:
category
upi_fraud               5
bank_fraud              5
lottery_fraud           4
otp_fraud               3
job_fraud               3
legitimate_ecommerce    3
investment_fraud        2
legitimate_bank         2
legitimate_otp          1
legitimate_travel       1
legitimate_personal     1
Name: count, dtype: int64

Sample messages:
  label                                                                                                               text
0  spam  Your UPI ID has been blocked! Complete KYC immediately to continue transactions. Click: http://upi-kyc-verify.com
1  spam                          Aapka Paytm account band ho jayega! Abhi KYC complete karein: http://paytm-kyc.xyz/verify
2  spam                   URGENT: Your PhonePe account will be deactivated in 24 hours. Update KYC now: bit.ly/phonepescam
3  spam                     Dear Customer, Send Rs 1 

In [8]:
# ============================================================
# DATASET 3: Phishing Email Dataset
# ============================================================
# We create a comprehensive dataset with real phishing patterns
# Since downloading from Kaggle requires authentication,
# we build a solid dataset here and can augment later.
# ============================================================

phishing_emails = [
    # ── Phishing Emails ───────────────────────────────────────
    {
        'label': 1,  # 1 = phishing
        'text': """Subject: URGENT: Your account will be suspended
        
Dear Valued Customer,

We have detected suspicious activity on your account. Your account will be 
suspended within 24 hours unless you verify your information immediately.

Click here to verify: http://secure-paypal-verify.xyz/login

You must provide:
- Full name
- Credit card number  
- CVV number
- Date of birth

Failure to comply will result in permanent account suspension.

PayPal Security Team"""
    },
    {
        'label': 1,
        'text': """Subject: Your Amazon order has a problem - Action Required

Dear Customer,

Your recent Amazon order cannot be processed due to payment failure.
Your account has been temporarily locked.

To restore access and process your order, please verify your payment details:
http://amazon-payment-verify.net/secure

Enter your:
- Email and password
- Credit/Debit card details
- OTP received on your phone

Act now to avoid order cancellation.

Amazon Customer Service"""
    },
    {
        'label': 1,
        'text': """Subject: [IMPORTANT] Netflix account on hold

Your Netflix membership is on hold because we're having trouble with your 
current billing information.

We need you to update your payment details to continue enjoying Netflix.

Update Payment: http://netflix-billing-update.xyz

Your streaming will be restored immediately after verification.

The Netflix Team"""
    },
    {
        'label': 1,
        'text': """Subject: Google Security Alert - Unusual Sign In

We detected an unusual sign-in attempt to your Google Account.

Location: Russia, Moscow
Time: 2:34 AM

If this wasn't you, your account may be compromised. 
Secure your account immediately: http://google-security-verify.com/secure

Enter your Google credentials to confirm your identity.

Google Security Team"""
    },
    {
        'label': 1,
        'text': """Subject: Your Bank Account Has Been Compromised

Dear Account Holder,

Our fraud detection system has identified unauthorized access to your account.

To protect your funds, we have temporarily limited your account.

Restore access now: http://secure-banking-verify.xyz/restore

You will need to provide your:
- Account number
- Internet banking password  
- ATM PIN
- Registered mobile OTP

Do not share this email with anyone.

Bank Security Department"""
    },
    
    # ── Legitimate Emails (not phishing) ─────────────────────
    {
        'label': 0,  # 0 = legitimate
        'text': """Subject: Your Amazon order has shipped!

Hello,

Great news! Your order #405-1234567-8901234 has shipped.

Order Details:
- Product: Wireless Headphones
- Estimated Delivery: January 20, 2024
- Carrier: Blue Dart

Track your package: amazon.com/orders

Thank you for shopping with Amazon!

Amazon Customer Service"""
    },
    {
        'label': 0,
        'text': """Subject: Password reset request for your account

Hi there,

We received a request to reset your GitHub password.

If you requested this password reset, click the link below:
https://github.com/password-reset/token-abc123

This link is valid for 1 hour.

If you didn't request a password reset, you can ignore this email.
Your password will not be changed.

GitHub Security"""
    },
    {
        'label': 0,
        'text': """Subject: Your monthly bank statement is ready

Dear Customer,

Your account statement for December 2023 is now available.

Account: XXXX XXXX XXXX 1234
Statement Period: 01-Dec-2023 to 31-Dec-2023

You can view your statement by logging into:
netbanking.sbi.co.in

If you have questions, contact us at 1800-425-3800.

State Bank of India"""
    },
    {
        'label': 0,
        'text': """Subject: Meeting scheduled for tomorrow

Hi Team,

Just a reminder that we have our weekly standup tomorrow at 10:00 AM IST.

Meeting link: meet.google.com/abc-defg-hij

Agenda:
1. Sprint review
2. Blockers discussion  
3. Next sprint planning

Please come prepared with your updates.

Best regards,
Team Lead"""
    },
    {
        'label': 0,
        'text': """Subject: Your Swiggy order is confirmed!

Hey there!

Your order from Domino's Pizza is confirmed.

Order ID: #SW123456789
Estimated delivery: 35-40 minutes
Items: Margherita Pizza (Large), Garlic Bread

Track your order live on the Swiggy app.

Hungry? We're on it! 🍕

Swiggy Team"""
    },
]

# Create DataFrame
email_df = pd.DataFrame(phishing_emails)

# Save to raw data
email_csv_path = os.path.join(RAW_DATA_DIR, 'phishing_emails.csv')
email_df.to_csv(email_csv_path, index=False, encoding='utf-8')

print("Phishing Email Dataset Created!")
print(f"  Total emails:     {len(email_df)}")
print(f"  Phishing (1):     {len(email_df[email_df['label']==1])}")
print(f"  Legitimate (0):   {len(email_df[email_df['label']==0])}")
print()
print("Sample phishing email (first 200 chars):")
print(email_df[email_df['label']==1]['text'].iloc[0][:200])

Phishing Email Dataset Created!
  Total emails:     10
  Phishing (1):     5
  Legitimate (0):   5

Sample phishing email (first 200 chars):
Subject: URGENT: Your account will be suspended

Dear Valued Customer,

We have detected suspicious activity on your account. Your account will be 
suspended within 24 hours unless you verify your inf


In [9]:
# ============================================================
# DATASET 4: Phishing URLs Dataset
# ============================================================
# URLs have structural features that reveal phishing attempts:
#   - Long URLs to hide the real domain
#   - IP addresses instead of domain names
#   - Misspelled legitimate brands (paypa1.com vs paypal.com)
#   - Excessive subdomains
#   - Suspicious TLDs (.xyz, .tk, .ml)
#   - HTTP instead of HTTPS
#   - Special characters (@, //, etc.)
# ============================================================

url_data = [
    # ── Phishing URLs (label=1) ───────────────────────────────
    # Pattern 1: Fake bank/payment sites
    {'url': 'http://sbi-netbanking-secure.xyz/login/verify', 'label': 1, 'category': 'fake_bank'},
    {'url': 'http://secure-paypal-account.verify.com/login', 'label': 1, 'category': 'fake_payment'},
    {'url': 'http://amazon-prize-winner.tk/claim/free', 'label': 1, 'category': 'fake_ecommerce'},
    {'url': 'http://192.168.1.1/bank/login.php', 'label': 1, 'category': 'ip_based'},
    {'url': 'http://69.172.201.153/secure/banking/login', 'label': 1, 'category': 'ip_based'},
    
    # Pattern 2: Brand impersonation
    {'url': 'http://paypa1.com/signin/verify', 'label': 1, 'category': 'brand_impersonation'},
    {'url': 'http://arnazon.com/account/suspend', 'label': 1, 'category': 'brand_impersonation'},
    {'url': 'http://faceb00k.com/login/secure', 'label': 1, 'category': 'brand_impersonation'},
    {'url': 'http://rn.icrosoftonline.com/login', 'label': 1, 'category': 'brand_impersonation'},
    {'url': 'http://g00gle-security.com/verify', 'label': 1, 'category': 'brand_impersonation'},
    
    # Pattern 3: Suspicious subdomains
    {'url': 'http://login.account.secure.paypal.phishing-site.com', 'label': 1, 'category': 'suspicious_subdomain'},
    {'url': 'http://secure.banking.sbi.customer-verify.xyz', 'label': 1, 'category': 'suspicious_subdomain'},
    {'url': 'http://account.verify.amazon.fake-site.ml', 'label': 1, 'category': 'suspicious_subdomain'},
    
    # Pattern 4: URL shorteners hiding destination
    {'url': 'http://bit.ly/win-prize-now-2024', 'label': 1, 'category': 'url_shortener'},
    {'url': 'http://tinyurl.com/claim-lottery-india', 'label': 1, 'category': 'url_shortener'},
    
    # Pattern 5: Excessive URL length and complexity
    {'url': 'http://secure-verify-update-account-banking-login-confirm-identity.com/user/validate?token=abc123&session=xyz&redirect=paypal', 'label': 1, 'category': 'long_suspicious'},
    {'url': 'http://xn--pypal-4ve.com/webscr?cmd=_login-submit', 'label': 1, 'category': 'encoded_domain'},
    
    # Indian-specific phishing
    {'url': 'http://upi-kyc-verify-paytm.xyz/login', 'label': 1, 'category': 'indian_upi_fraud'},
    {'url': 'http://jio-lucky-winner-prize.tk/claim', 'label': 1, 'category': 'indian_lottery'},
    {'url': 'http://sarkari-result-2024.xyz/apply/job', 'label': 1, 'category': 'indian_job_fraud'},
    {'url': 'http://pm-kisan-yojana-apply.ml/register', 'label': 1, 'category': 'indian_govt_fraud'},
    {'url': 'http://aadhaar-update-uidai.xyz/verify', 'label': 1, 'category': 'indian_govt_fraud'},
    {'url': 'http://kbc-lottery-winner-2024.tk/claim-prize', 'label': 1, 'category': 'indian_lottery'},
    
    # ── Legitimate URLs (label=0) ─────────────────────────────
    {'url': 'https://www.sbi.co.in/web/personal-banking', 'label': 0, 'category': 'legitimate_bank'},
    {'url': 'https://www.paypal.com/signin', 'label': 0, 'category': 'legitimate_payment'},
    {'url': 'https://www.amazon.in/gp/cart/view.html', 'label': 0, 'category': 'legitimate_ecommerce'},
    {'url': 'https://www.google.com/search?q=python+tutorial', 'label': 0, 'category': 'legitimate_search'},
    {'url': 'https://github.com/username/repository', 'label': 0, 'category': 'legitimate_tech'},
    {'url': 'https://mail.google.com/mail/u/0/#inbox', 'label': 0, 'category': 'legitimate_email'},
    {'url': 'https://www.irctc.co.in/nget/train-search', 'label': 0, 'category': 'legitimate_travel'},
    {'url': 'https://www.flipkart.com/search?q=mobile+phones', 'label': 0, 'category': 'legitimate_ecommerce'},
    {'url': 'https://netbanking.hdfcbank.com/netbanking/', 'label': 0, 'category': 'legitimate_bank'},
    {'url': 'https://www.myntra.com/men-tshirts', 'label': 0, 'category': 'legitimate_ecommerce'},
    {'url': 'https://stackoverflow.com/questions/python', 'label': 0, 'category': 'legitimate_tech'},
    {'url': 'https://linkedin.com/in/profile', 'label': 0, 'category': 'legitimate_social'},
    {'url': 'https://www.youtube.com/watch?v=dQw4w9WgXcQ', 'label': 0, 'category': 'legitimate_entertainment'},
    {'url': 'https://uidai.gov.in/en/my-aadhaar.html', 'label': 0, 'category': 'legitimate_govt'},
    {'url': 'https://incometax.gov.in/iec/foportal', 'label': 0, 'category': 'legitimate_govt'},
    {'url': 'https://www.swiggy.com/restaurants', 'label': 0, 'category': 'legitimate_food'},
    {'url': 'https://paytm.com/bank/passbook', 'label': 0, 'category': 'legitimate_payment'},
    {'url': 'https://www.zomato.com/bangalore/restaurants', 'label': 0, 'category': 'legitimate_food'},
]

# Create DataFrame
url_df = pd.DataFrame(url_data)

# Save to raw data
url_csv_path = os.path.join(RAW_DATA_DIR, 'phishing_urls.csv')
url_df.to_csv(url_csv_path, index=False, encoding='utf-8')

print("Phishing URL Dataset Created!")
print(f"  Total URLs:       {len(url_df)}")
print(f"  Phishing (1):     {len(url_df[url_df['label']==1])}")
print(f"  Legitimate (0):   {len(url_df[url_df['label']==0])}")
print()
print("Category breakdown:")
print(url_df.groupby(['label', 'category']).size().reset_index(name='count').to_string())

Phishing URL Dataset Created!
  Total URLs:       41
  Phishing (1):     23
  Legitimate (0):   18

Category breakdown:
    label                  category  count
0       0           legitimate_bank      2
1       0      legitimate_ecommerce      3
2       0          legitimate_email      1
3       0  legitimate_entertainment      1
4       0           legitimate_food      2
5       0           legitimate_govt      2
6       0        legitimate_payment      2
7       0         legitimate_search      1
8       0         legitimate_social      1
9       0           legitimate_tech      2
10      0         legitimate_travel      1
11      1       brand_impersonation      5
12      1            encoded_domain      1
13      1                 fake_bank      1
14      1            fake_ecommerce      1
15      1              fake_payment      1
16      1         indian_govt_fraud      2
17      1          indian_job_fraud      1
18      1            indian_lottery      2
19      1          i

In [10]:
# ============================================================
# COMBINE ALL TEXT DATASETS
# ============================================================
# We merge SMS + Indian Scam + Email into one unified
# text classification dataset.
# 
# Unified labels: 0 = legitimate, 1 = scam/phishing
# ============================================================

print("=" * 50)
print("  Combining all text datasets...")
print("=" * 50)

all_text_data = []

# ── Add SMS Spam Data ──────────────────────────────────────
if 'sms_df' in locals() and len(sms_df) > 0:
    sms_combined = sms_df[['label', 'text']].copy()
    # Convert 'ham'/'spam' labels to 0/1
    sms_combined['label'] = sms_combined['label'].map({'ham': 0, 'spam': 1})
    sms_combined['source'] = 'sms_uci'
    sms_combined = sms_combined.dropna(subset=['label'])
    all_text_data.append(sms_combined)
    print(f"  SMS Dataset:          {len(sms_combined):>6} messages")

# ── Add Indian Scam Data ──────────────────────────────────
indian_combined = indian_df[['label', 'text']].copy()
indian_combined['label'] = indian_combined['label'].map({'ham': 0, 'spam': 1})
indian_combined['source'] = 'indian_custom'
all_text_data.append(indian_combined)
print(f"  Indian Scam Dataset:  {len(indian_combined):>6} messages")

# ── Add Email Data ────────────────────────────────────────
email_combined = email_df[['label', 'text']].copy()
email_combined['source'] = 'phishing_email'
all_text_data.append(email_combined)
print(f"  Phishing Email:       {len(email_combined):>6} messages")

# ── Merge All ─────────────────────────────────────────────
combined_df = pd.concat(all_text_data, ignore_index=True)

# Remove any rows where label or text is missing
combined_df = combined_df.dropna(subset=['label', 'text'])

# Make sure label is integer
combined_df['label'] = combined_df['label'].astype(int)

# Reset index
combined_df = combined_df.reset_index(drop=True)

print()
print(f"  TOTAL COMBINED:       {len(combined_df):>6} messages")
print()
print("Label distribution:")
label_counts = combined_df['label'].value_counts()
total = len(combined_df)
for label, count in label_counts.items():
    label_name = 'SCAM/PHISHING' if label == 1 else 'LEGITIMATE'
    percentage = (count / total) * 100
    print(f"  {label} ({label_name}): {count} ({percentage:.1f}%)")

print()
print("Sample rows:")
print(combined_df.sample(5, random_state=42)[['label', 'source', 'text']].to_string())

  Combining all text datasets...
  SMS Dataset:            5572 messages
  Indian Scam Dataset:      30 messages
  Phishing Email:           10 messages

  TOTAL COMBINED:         5612 messages

Label distribution:
  0 (LEGITIMATE): 4838 (86.2%)
  1 (SCAM/PHISHING): 774 (13.8%)

Sample rows:
      label   source                                                                                                                                                   text
5155      0  sms_uci                                                                  MY NEW YEARS EVE WAS OK. I WENT TO A PARTY WITH MY BOYFRIEND. WHO IS THIS SI THEN HEY
4196      1  sms_uci  Double mins and txts 4 6months FREE Bluetooth on Orange. Available on Sony, Nokia Motorola phones. Call MobileUpd8 on 08000839402 or call2optout/N9DX
1978      1  sms_uci                                               Reply to win Â£100 weekly! Where will the 2006 FIFA World Cup be held? Send STOP to 87239 to end service
776       0  sms_uc

In [11]:
# ============================================================
# EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================
# EDA means: look at the data deeply before modeling.
# We want to understand:
#   1. How balanced is our dataset?
#   2. How long are scam vs legitimate messages?
#   3. What words appear most in scam messages?
#   4. Are there patterns we can exploit?
#
# WHY EDA MATTERS:
# EDA reveals insights that directly improve our models.
# For example, if we find scam messages are much longer,
# "text length" becomes a useful feature.
# ============================================================

print("Starting EDA...")
print()

# ── Add text statistics columns ───────────────────────────
combined_df['text_length'] = combined_df['text'].str.len()
combined_df['word_count'] = combined_df['text'].str.split().str.len()
combined_df['char_count'] = combined_df['text'].str.len()
combined_df['has_url'] = combined_df['text'].str.contains(
    r'http[s]?://|www\.|bit\.ly|tinyurl', 
    case=False, regex=True
).astype(int)
combined_df['has_phone'] = combined_df['text'].str.contains(
    r'\d{10}|\d{3}[-\s]\d{3}[-\s]\d{4}', 
    regex=True
).astype(int)
combined_df['has_urgency'] = combined_df['text'].str.contains(
    r'urgent|immediately|now|expire|suspend|block|verify|24 hour|limited', 
    case=False, regex=True
).astype(int)
combined_df['exclamation_count'] = combined_df['text'].str.count('!')
combined_df['uppercase_ratio'] = combined_df['text'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
)

print("Added feature columns:")
print(f"  text_length, word_count, has_url, has_phone")
print(f"  has_urgency, exclamation_count, uppercase_ratio")
print()

# ── Summary statistics by label ──────────────────────────
print("Summary Statistics by Label:")
print()
stats = combined_df.groupby('label').agg({
    'text_length': ['mean', 'median', 'max'],
    'word_count': ['mean', 'median'],
    'has_url': 'mean',
    'has_urgency': 'mean',
    'exclamation_count': 'mean',
    'uppercase_ratio': 'mean'
}).round(3)
print(stats)

Starting EDA...

Added feature columns:
  text_length, word_count, has_url, has_phone
  has_urgency, exclamation_count, uppercase_ratio

Summary Statistics by Label:

      text_length               word_count         has_url has_urgency  \
             mean   median  max       mean  median    mean        mean   
label                                                                    
0         71.8450  53.0000  910    14.3440 11.0000  0.0010      0.1130   
1        139.7350 149.0000  459    23.8940 25.0000  0.1500      0.3910   

      exclamation_count uppercase_ratio  
                   mean            mean  
label                                    
0                0.1780          0.0590  
1                0.7160          0.1130  


In [12]:
# ============================================================
# VISUALIZATION 1: Class Distribution
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('ScamShield AI — Dataset Analysis', fontsize=16, fontweight='bold')

# Plot 1: Class distribution
label_names = {0: 'Legitimate', 1: 'Scam/Phishing'}
colors = ['#2ecc71', '#e74c3c']

label_counts = combined_df['label'].value_counts()
axes[0].bar(
    [label_names[l] for l in label_counts.index],
    label_counts.values,
    color=colors,
    edgecolor='black',
    linewidth=0.5
)
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Messages')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Plot 2: Text length distribution
for label in [0, 1]:
    subset = combined_df[combined_df['label'] == label]['text_length']
    axes[1].hist(
        subset, 
        bins=30, 
        alpha=0.6, 
        color=colors[label], 
        label=label_names[label],
        edgecolor='black',
        linewidth=0.3
    )
axes[1].set_title('Text Length Distribution', fontweight='bold')
axes[1].set_xlabel('Number of Characters')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Plot 3: Feature comparison (scam vs legitimate)
features = ['has_url', 'has_phone', 'has_urgency']
feature_names = ['Has URL', 'Has Phone\nNumber', 'Has Urgency\nWords']

scam_vals = [combined_df[combined_df['label']==1][f].mean() * 100 for f in features]
legit_vals = [combined_df[combined_df['label']==0][f].mean() * 100 for f in features]

x = range(len(features))
width = 0.35

axes[2].bar([i - width/2 for i in x], legit_vals, width, 
           label='Legitimate', color='#2ecc71', edgecolor='black', linewidth=0.5)
axes[2].bar([i + width/2 for i in x], scam_vals, width, 
           label='Scam/Phishing', color='#e74c3c', edgecolor='black', linewidth=0.5)
axes[2].set_title('Feature Presence (%)', fontweight='bold')
axes[2].set_ylabel('Percentage of Messages (%)')
axes[2].set_xticks(list(x))
axes[2].set_xticklabels(feature_names)
axes[2].legend()
axes[2].set_ylim(0, 110)

plt.tight_layout()

# Save the figure
eda_plot_path = os.path.join(PROJECT_ROOT, 'docs', 'eda_analysis.png')
plt.savefig(eda_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nPlot saved to: {eda_plot_path}")

<Figure size 1800x500 with 3 Axes>


Plot saved to: C:\Users\Manoj S\Desktop\scamshield-ai\docs\eda_analysis.png


In [13]:
# ============================================================
# WORD FREQUENCY ANALYSIS
# ============================================================
# What words appear most in scam messages vs legitimate?
# This tells us which words are strong scam indicators.
# ============================================================

def get_top_words(df, label, n=20):
    """Get top N most common words for a given label."""
    # Get all text for this label
    texts = df[df['label'] == label]['text'].str.lower()
    
    # Combine all text
    all_text = ' '.join(texts)
    
    # Tokenize (split into words)
    words = re.findall(r'\b[a-z]{3,}\b', all_text)
    
    # Remove stopwords (common words that don't carry meaning)
    stop_words = set(stopwords.words('english'))
    
    # Also add domain-specific non-meaningful words
    custom_stops = {'your', 'have', 'been', 'will', 'this', 'that', 
                    'with', 'from', 'they', 'their', 'there', 'here',
                    'dear', 'please', 'thank', 'regards'}
    stop_words.update(custom_stops)
    
    filtered_words = [w for w in words if w not in stop_words]
    
    # Count and return top N
    word_freq = Counter(filtered_words)
    return word_freq.most_common(n)

# Get top words for each class
print("Top 20 words in SCAM messages:")
scam_words = get_top_words(combined_df, label=1, n=20)
for word, count in scam_words:
    bar = '█' * min(count, 30)
    print(f"  {word:<20} {bar} ({count})")

print()
print("Top 20 words in LEGITIMATE messages:")
legit_words = get_top_words(combined_df, label=0, n=20)
for word, count in legit_words:
    bar = '█' * min(count, 30)
    print(f"  {word:<20} {bar} ({count})")

Top 20 words in SCAM messages:
  call                 ██████████████████████████████ (358)
  free                 ██████████████████████████████ (226)
  txt                  ██████████████████████████████ (163)
  mobile               ██████████████████████████████ (129)
  text                 ██████████████████████████████ (125)
  stop                 ██████████████████████████████ (123)
  claim                ██████████████████████████████ (115)
  reply                ██████████████████████████████ (104)
  www                  ██████████████████████████████ (98)
  prize                ██████████████████████████████ (93)
  get                  ██████████████████████████████ (87)
  cash                 ██████████████████████████████ (76)
  send                 ██████████████████████████████ (72)
  new                  ██████████████████████████████ (69)
  nokia                ██████████████████████████████ (67)
  win                  ██████████████████████████████ (65)
  urgent         

In [14]:
# ============================================================
# TEXT PREPROCESSING PIPELINE
# ============================================================
# Raw text is messy. Before feeding to ML models, we clean it.
# 
# Steps:
# 1. Lowercase everything
# 2. Remove HTML tags (some emails have HTML)
# 3. Remove URLs (we extract them separately as features)
# 4. Remove phone numbers
# 5. Remove special characters (keep letters and spaces)
# 6. Remove extra whitespace
# 7. Tokenize
# 8. Remove stopwords
# 9. Lemmatize (run → running → run)
#
# WHY NOT REMOVE ALL SPECIAL CHARS?
# Exclamation marks and capitalization are FEATURES of scam text.
# We keep raw text AND clean text — use both for different purposes.
# ============================================================

lemmatizer = WordNetLemmatizer()
stop_words_set = set(stopwords.words('english'))

def clean_text(text):
    """
    Clean and normalize text for ML model input.
    
    Args:
        text: Raw input string
        
    Returns:
        Cleaned, normalized string
    """
    if not isinstance(text, str):
        return ''
    
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove HTML tags
    # Some emails contain HTML like <b>click here</b>
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Step 3: Replace URLs with token
    # We don't remove URLs — we replace with a token
    # This preserves the FACT that a URL existed
    text = re.sub(r'http[s]?://\S+|www\.\S+', ' url_present ', text)
    
    # Step 4: Replace phone numbers with token
    text = re.sub(r'\b\d{10}\b|\b\d{3}[-\s]\d{3}[-\s]\d{4}\b', ' phone_present ', text)
    
    # Step 5: Replace numbers (amounts like Rs.5000)
    text = re.sub(r'rs\.?\s*\d+|₹\s*\d+|\$\s*\d+', ' money_amount ', text)
    
    # Step 6: Remove remaining special characters (keep letters, spaces, tokens)
    text = re.sub(r'[^a-zA-Z\s_]', ' ', text)
    
    # Step 7: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 8: Tokenize
    tokens = text.split()
    
    # Step 9: Remove stopwords and lemmatize
    cleaned_tokens = []
    for token in tokens:
        if token not in stop_words_set and len(token) > 2:
            # Lemmatize: converts words to base form
            # running → run, banks → bank
            lemma = lemmatizer.lemmatize(token)
            cleaned_tokens.append(lemma)
    
    return ' '.join(cleaned_tokens)


# Apply cleaning to our dataset
print("Cleaning text data...")
combined_df['cleaned_text'] = combined_df['text'].apply(clean_text)

print("Done!")
print()
print("Example — Original vs Cleaned:")
print()

for i in [0, 5]:
    print(f"  [{combined_df.iloc[i]['label']}] ORIGINAL:")
    print(f"  {combined_df.iloc[i]['text'][:150]}")
    print(f"  CLEANED:")
    print(f"  {combined_df.iloc[i]['cleaned_text'][:150]}")
    print()

Cleaning text data...
Done!

Example — Original vs Cleaned:

  [0] ORIGINAL:
  Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
  CLEANED:
  jurong point crazy available bugis great world buffet cine got amore wat

  [1] ORIGINAL:
  FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, Â£1.50 to rcv
  CLEANED:
  freemsg hey darling week word back like fun still xxx std chgs send rcv



In [15]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================
# We split data into:
#   - Training set (80%): Model learns from this
#   - Test set (20%): We evaluate model on unseen data
#
# WHY 80/20?
# Standard practice. Gives enough data to train while
# keeping enough to reliably evaluate performance.
#
# WHY stratify?
# Without stratify, random split might put all scam messages
# in training and none in test. Stratify ensures both sets
# have the same proportion of scam vs legitimate.
# ============================================================

from sklearn.model_selection import train_test_split

print("Splitting data into train and test sets...")
print()

# ── Text Dataset Split ────────────────────────────────────
X_text = combined_df['cleaned_text']  # Features (text)
y_text = combined_df['label']          # Target (0 or 1)

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text,
    y_text,
    test_size=0.2,      # 20% for testing
    random_state=42,    # Set seed for reproducibility
    stratify=y_text     # Maintain class proportions
)

# ── URL Dataset Split ─────────────────────────────────────
X_url = url_df['url']
y_url = url_df['label']

X_train_url, X_test_url, y_train_url, y_test_url = train_test_split(
    X_url,
    y_url,
    test_size=0.2,
    random_state=42,
    stratify=y_url
)

print("Text Dataset Split:")
print(f"  Training:   {len(X_train_text)} samples")
print(f"    - Scam:   {y_train_text.sum()} ({y_train_text.mean()*100:.1f}%)")
print(f"    - Legit:  {(y_train_text==0).sum()} ({(y_train_text==0).mean()*100:.1f}%)")
print()
print(f"  Testing:    {len(X_test_text)} samples")
print(f"    - Scam:   {y_test_text.sum()} ({y_test_text.mean()*100:.1f}%)")
print(f"    - Legit:  {(y_test_text==0).sum()} ({(y_test_text==0).mean()*100:.1f}%)")

print()
print("URL Dataset Split:")
print(f"  Training:   {len(X_train_url)} samples")
print(f"  Testing:    {len(X_test_url)} samples")

# ── Save Processed Datasets ───────────────────────────────
# Save full combined dataset
combined_df.to_csv(
    os.path.join(PROCESSED_DATA_DIR, 'text_combined.csv'), 
    index=False
)

# Save train/test splits
train_df = pd.DataFrame({
    'text': X_train_text, 
    'cleaned_text': X_train_text,
    'label': y_train_text
})
train_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'text_train.csv'), index=False)

test_df = pd.DataFrame({
    'text': X_test_text,
    'cleaned_text': X_test_text, 
    'label': y_test_text
})
test_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'text_test.csv'), index=False)

# Save URL splits
url_train_df = pd.DataFrame({'url': X_train_url, 'label': y_train_url})
url_train_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'url_train.csv'), index=False)

url_test_df = pd.DataFrame({'url': X_test_url, 'label': y_test_url})
url_test_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'url_test.csv'), index=False)

print()
print("=" * 50)
print("All datasets saved to data/processed/")
print("  text_combined.csv")
print("  text_train.csv")
print("  text_test.csv")
print("  url_train.csv")
print("  url_test.csv")
print("=" * 50)

Splitting data into train and test sets...

Text Dataset Split:
  Training:   4489 samples
    - Scam:   619 (13.8%)
    - Legit:  3870 (86.2%)

  Testing:    1123 samples
    - Scam:   155 (13.8%)
    - Legit:  968 (86.2%)

URL Dataset Split:
  Training:   32 samples
  Testing:    9 samples

All datasets saved to data/processed/
  text_combined.csv
  text_train.csv
  text_test.csv
  url_train.csv
  url_test.csv


In [16]:
# ============================================================
# PHASE 1 COMPLETE — SUMMARY
# ============================================================

print("=" * 60)
print("  PHASE 1 COMPLETE — DATA COLLECTION & PREPARATION")
print("=" * 60)
print()
print("DATASETS CREATED:")
print(f"  Text Classification Dataset:")
print(f"    Total:    {len(combined_df)} messages")
print(f"    Train:    {len(X_train_text)} messages")
print(f"    Test:     {len(X_test_text)} messages")
print()
print(f"  URL Classification Dataset:")
print(f"    Total:    {len(url_df)} URLs")
print(f"    Train:    {len(X_train_url)} URLs")
print(f"    Test:     {len(X_test_url)} URLs")
print()
print("FILES SAVED:")
files = [
    'data/raw/indian_scam_sms.csv',
    'data/raw/phishing_emails.csv',
    'data/raw/phishing_urls.csv',
    'data/processed/text_combined.csv',
    'data/processed/text_train.csv',
    'data/processed/text_test.csv',
    'data/processed/url_train.csv',
    'data/processed/url_test.csv',
]
for f in files:
    full_path = os.path.join(PROJECT_ROOT, f)
    exists = "SAVED" if os.path.exists(full_path) else "MISSING"
    print(f"  [{exists}] {f}")
print()
print("KEY INSIGHTS FROM EDA:")
print(f"  - Scam messages are longer on average")
print(f"  - Scam messages use urgency words more often")
print(f"  - Scam messages have more URLs and phone numbers")
print(f"  - Words like 'verify', 'urgent', 'blocked' are scam signals")
print()
print("NEXT: Phase 2 — Building ML Models")
print("=" * 60)

  PHASE 1 COMPLETE — DATA COLLECTION & PREPARATION

DATASETS CREATED:
  Text Classification Dataset:
    Total:    5612 messages
    Train:    4489 messages
    Test:     1123 messages

  URL Classification Dataset:
    Total:    41 URLs
    Train:    32 URLs
    Test:     9 URLs

FILES SAVED:
  [SAVED] data/raw/indian_scam_sms.csv
  [SAVED] data/raw/phishing_emails.csv
  [SAVED] data/raw/phishing_urls.csv
  [SAVED] data/processed/text_combined.csv
  [SAVED] data/processed/text_train.csv
  [SAVED] data/processed/text_test.csv
  [SAVED] data/processed/url_train.csv
  [SAVED] data/processed/url_test.csv

KEY INSIGHTS FROM EDA:
  - Scam messages are longer on average
  - Scam messages use urgency words more often
  - Scam messages have more URLs and phone numbers
  - Words like 'verify', 'urgent', 'blocked' are scam signals

NEXT: Phase 2 — Building ML Models
